# New collection

`feature_counts` is how many points each feature has. Keys are strings, so subject `7` is `"7"`. `vectors_config` is the vector width plus the distance you used when you built the vectors. The file does not store that distance.

`collection_setup` scans the file once and returns both arguments. Pass them into `create_collection`. The same call works for CSV, JSON, JSONL, NumPy `.npz`, and Parquet.

CSV and JSON are the formats to start with. Parquet is the one to use once the table is large. Hugging Face datasets are Parquet. A memmap is for a matrix that does not fit in memory: you open the arrays, then call `count_features` and `vector_params`.

In [ ]:
import json
from pathlib import Path

import numpy as np
from qdrant_client.models import Distance

from qdrant_custom_sharding import collection_setup, count_features, vector_params

sample = Path("sample")
sample.mkdir(exist_ok=True)
rows = [
    {"subject_id": 7, "vector": [0.1, 0.2, 0.3, 0.4]},
    {"subject_id": 7, "vector": [0.2, 0.2, 0.2, 0.2]},
    {"subject_id": 8, "vector": [0.0, 0.1, 0.0, 0.1]},
    {"subject_id": 9, "vector": [0.4, 0.3, 0.2, 0.1]},
]


## CSV

One column is the feature. The vector column is a JSON list. A spreadsheet with `v0, v1, v2, v3` can pass those names as a list instead of `"vector"`.

In [ ]:
csv_path = sample / "people.csv"
lines = ["subject_id,vector"]
for row in rows:
    lines.append(f"{row['subject_id']},\"{json.dumps(row['vector'])}\"")
csv_path.write_text("\n".join(lines) + "\n", encoding="utf-8")

setup = collection_setup(csv_path, "subject_id", "vector", Distance.COSINE)
setup


That `setup` dict is what `create_collection` needs.

```python
client.sharding.create_collection(
    "people",
    "subject_id",
    feature_type="integer",
    n_slots=48,
    **setup,
)
```

## JSON and JSONL

JSON is a list of objects. JSONL is one object per line. The field names are the same.

In [ ]:
json_path = sample / "people.json"
json_path.write_text(json.dumps(rows), encoding="utf-8")
jsonl_path = sample / "people.jsonl"
jsonl_path.write_text("".join(json.dumps(row) + "\n" for row in rows), encoding="utf-8")

collection_setup(json_path, "subject_id", "vector", Distance.COSINE)
collection_setup(jsonl_path, "subject_id", "vector", Distance.COSINE)


## NumPy `.npz`

Save two arrays under the field names. This is the file to use when the vectors are already a matrix.

In [ ]:
npz_path = sample / "people.npz"
np.savez(
    npz_path,
    subject_id=np.array([row["subject_id"] for row in rows]),
    vector=np.array([row["vector"] for row in rows], dtype=np.float32),
)
collection_setup(npz_path, "subject_id", "vector", Distance.COSINE)


## Memmap

A memmap is two arrays on disk. Open them, then use the same two functions. You pass the shape, because the file itself does not store it.

In [ ]:
ids = np.memmap(sample / "ids.i64", dtype=np.int64, mode="w+", shape=(len(rows),))
vectors = np.memmap(
    sample / "vectors.f32", dtype=np.float32, mode="w+", shape=(len(rows), 4)
)
ids[:] = [row["subject_id"] for row in rows]
vectors[:] = [row["vector"] for row in rows]
ids.flush()
vectors.flush()

opened_ids = np.memmap(sample / "ids.i64", dtype=np.int64, mode="r", shape=(len(rows),))
opened_vectors = np.memmap(
    sample / "vectors.f32", dtype=np.float32, mode="r", shape=(len(rows), 4)
)
count_features(opened_ids), vector_params(opened_vectors, Distance.COSINE)


## Parquet

Same call. Install `pyarrow` first. Polars and Hugging Face write this format too. Point `collection_setup` at the `.parquet` file.

In [ ]:
parquet_path = sample / "people.parquet"
try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except ImportError:
    print("pip install pyarrow")
else:
    table = pa.table(
        {
            "subject_id": [row["subject_id"] for row in rows],
            "vector": [row["vector"] for row in rows],
        }
    )
    pq.write_table(table, parquet_path)
    print(collection_setup(parquet_path, "subject_id", "vector", Distance.COSINE))
